# Olive Counting System - Training Notebook
**Based on Chen et al. (2017): "Counting Apples and Oranges with Deep Learning"**

## 3-Stage Pipeline:
1. **FCN Blob Detector**: Segments fruit blobs from background (pixel-level segmentation)
2. **CNN Counter**: Counts number of fruits in each segmented blob
3. **Linear Regression**: Maps intermediate count to final count

---

## 1. Setup and Imports

In [2]:
import sys
import os
from pathlib import Path
import torch

# Add Main directory to Python path
main_dir = os.path.dirname(os.path.abspath("__file__"))
if main_dir not in sys.path:
    sys.path.insert(0, main_dir)

# Import modules
from models import FCNBlobDetector, CNNCountingNetwork, LinearRegressionCorrection
from dataset import OliveSegmentationDataset, create_data_loaders
from pipeline import OliveCountingPipeline

print("✓ Imports successful")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

✓ Imports successful
PyTorch version: 2.10.0+cpu
CUDA available: False


## 2. Initialize Pipeline

In [3]:
# Detect device (GPU if available, otherwise CPU)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

# Initialize the 3-stage pipeline
pipeline = OliveCountingPipeline(device=device)
print("\n✓ Pipeline initialized!")

Using device: cpu


c:\Users\Welcome\OneDrive\Desktop\FYP MATERIAL\patchify\.venv\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\Welcome\OneDrive\Desktop\FYP MATERIAL\patchify\.venv\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)



✓ Pipeline initialized!


## 3. Prepare Dataset

Loading image-mask pairs from:
- Images: `patchify/` directory
- Masks: `patchify_mask/` directory

In [4]:
# Set data directories (relative to workspace root)
image_dir = r'C:\Users\Welcome\OneDrive\Desktop\FYP MATERIAL\patchify'
mask_dir = r'C:\Users\Welcome\OneDrive\Desktop\FYP MATERIAL\patchify_mask'

print(f"Image directory: {image_dir}")
print(f"Mask directory:  {mask_dir}")

Image directory: C:\Users\Welcome\OneDrive\Desktop\FYP MATERIAL\patchify
Mask directory:  C:\Users\Welcome\OneDrive\Desktop\FYP MATERIAL\patchify_mask


In [5]:
# Create data loaders (80% train, 20% validation)
train_loader, val_loader = create_data_loaders(
    image_dir=image_dir,
    mask_dir=mask_dir,
    batch_size=8,
    train_split=0.8
)

print(f"\n✓ Data loaders created")
print(f"  Training batches: {len(train_loader)}")
print(f"  Validation batches: {len(val_loader)}")

Found 909 images in C:\Users\Welcome\OneDrive\Desktop\FYP MATERIAL\patchify
Found 909 matching image-mask pairs

Dataset split:
  Training: 727 images
  Validation: 182 images

✓ Data loaders created
  Training batches: 91
  Validation batches: 23


## 4. Stage 1: Train FCN Segmentation

Training the FCN-8s network to segment olive blobs from background.

**Note:** Training will take several hours on CPU. Consider:
- Reducing epochs for testing (e.g., `epochs=5`)
- Using GPU for faster training
- The best model will be saved as `fcn_segmenter_best.pth`

In [12]:
# Train Stage 1: FCN Blob Detector (full dataset, 1 epoch)
import torch.nn as nn
import time

print("============================================================")
print("STAGE 1: Training FCN Blob Detector")
print(f"Training on ALL {len(train_loader)} batches ({len(train_loader)*8} images)")
print("============================================================\n")

pipeline.fcn_segmenter.train()
optimizer = torch.optim.Adam(pipeline.fcn_segmenter.parameters(), lr=1e-4)
criterion = nn.CrossEntropyLoss()

start_time = time.time()
total_batches = len(train_loader)

for batch_idx, (images, masks) in enumerate(train_loader):
    batch_start = time.time()
    images = images.to(device)
    masks = masks.to(device).long()

    optimizer.zero_grad()
    outputs = pipeline.fcn_segmenter(images)
    loss = criterion(outputs, masks)
    loss.backward()
    optimizer.step()

    batch_time = time.time() - batch_start
    elapsed = time.time() - start_time
    eta = (elapsed / (batch_idx + 1)) * (total_batches - batch_idx - 1)

    # Print progress every 10 batches
    if (batch_idx + 1) % 10 == 0 or batch_idx == 0:
        print(f"Batch {batch_idx+1}/{total_batches} - Loss: {loss.item():.4f} - {batch_time:.1f}s/batch - ETA: {eta/60:.1f}min")

total_time = time.time() - start_time
print(f"\n✓ Training complete in {total_time/60:.1f} minutes")

# Save model
torch.save(pipeline.fcn_segmenter.state_dict(), 'fcn_segmenter_best.pth')
print("✓ Model saved to fcn_segmenter_best.pth")

STAGE 1: Training FCN Blob Detector
Training on ALL 91 batches (728 images)

Batch 1/91 - Loss: 0.2433 - 21.5s/batch - ETA: 32.4min
Batch 10/91 - Loss: 0.1650 - 23.2s/batch - ETA: 30.7min
Batch 20/91 - Loss: 0.1274 - 25.2s/batch - ETA: 28.3min
Batch 30/91 - Loss: 0.1058 - 25.1s/batch - ETA: 24.8min
Batch 40/91 - Loss: 0.0915 - 26.0s/batch - ETA: 20.9min
Batch 50/91 - Loss: 0.0841 - 24.7s/batch - ETA: 16.9min
Batch 60/91 - Loss: 0.1441 - 25.1s/batch - ETA: 12.8min
Batch 70/91 - Loss: 0.1943 - 25.0s/batch - ETA: 8.7min
Batch 80/91 - Loss: 0.1835 - 25.2s/batch - ETA: 4.6min
Batch 90/91 - Loss: 0.1398 - 24.8s/batch - ETA: 0.4min

✓ Training complete in 37.8 minutes
✓ Model saved to fcn_segmenter_best.pth


## 5. Stage 2: Train CNN Counter (Optional)

**Requirements:**
- JSON annotation files with blob coordinates
- Extracted blob images from segmentation results

⚠️ **Currently unavailable** - requires ground truth blob annotations

In [ ]:
# Stage 2 training (requires blob dataset)
# Uncomment and configure when blob annotations are available

# blob_loader = load_blob_dataset(...)  # Need to implement
# pipeline.train_stage2_counting(
#     blob_loader=blob_loader,
#     epochs=50,
#     learning_rate=1e-3
# )

print("⚠️ Stage 2 requires blob annotations (not available)")

## 6. Stage 3: Train Linear Regression (Optional)

**Requirements:**
- Ground truth olive counts for each image
- Stage 2 CNN predictions

⚠️ **Currently unavailable** - requires ground truth count labels

In [ ]:
# Stage 3 training (requires count labels)
# Uncomment when ground truth counts are available

# X_train = [...]  # CNN predictions
# y_train = [...]  # Ground truth counts
# pipeline.train_stage3_regression(X_train, y_train)

print("⚠️ Stage 3 requires ground truth count labels (not available)")

## 7. Prediction on New Images

Use the trained pipeline to predict olive counts on new images.

In [ ]:
# Test the trained model on validation images
import matplotlib.pyplot as plt
import numpy as np

# Load the saved model
pipeline.fcn_segmenter.load_state_dict(torch.load('fcn_segmenter_best.pth', map_location=device))
pipeline.fcn_segmenter.eval()
print("✓ Model loaded from fcn_segmenter_best.pth")

# Get a batch of validation images
val_images, val_masks = next(iter(val_loader))
val_images = val_images.to(device)

# Predict
with torch.no_grad():
    predictions = pipeline.fcn_segmenter(val_images)
    pred_masks = torch.argmax(predictions, dim=1).cpu().numpy()

val_masks_np = val_masks.numpy()

# Denormalize images for display
mean = np.array([0.485, 0.456, 0.406])
std = np.array([0.229, 0.224, 0.225])

# Show 4 results: Original | Ground Truth Mask | Predicted Mask
num_show = min(4, len(val_images))
fig, axes = plt.subplots(num_show, 3, figsize=(12, 4*num_show))

for i in range(num_show):
    img = val_images[i].cpu().numpy().transpose(1, 2, 0)
    img = (img * std + mean).clip(0, 1)
    
    axes[i, 0].imshow(img)
    axes[i, 0].set_title("Original Image")
    axes[i, 0].axis('off')
    
    axes[i, 1].imshow(val_masks_np[i], cmap='gray')
    axes[i, 1].set_title("Ground Truth Mask")
    axes[i, 1].axis('off')
    
    axes[i, 2].imshow(pred_masks[i], cmap='gray')
    axes[i, 2].set_title("Predicted Mask")
    axes[i, 2].axis('off')

plt.tight_layout()
plt.savefig('segmentation_results.png', dpi=150)
plt.show()

# Calculate IoU on full validation set
total_iou = 0
num_samples = 0
with torch.no_grad():
    for images, masks in val_loader:
        images = images.to(device)
        outputs = pipeline.fcn_segmenter(images)
        preds = torch.argmax(outputs, dim=1).cpu().numpy()
        masks_np = masks.numpy()
        
        for p, m in zip(preds, masks_np):
            intersection = ((p == 1) & (m == 1)).sum()
            union = ((p == 1) | (m == 1)).sum()
            if union > 0:
                total_iou += intersection / union
            num_samples += 1

avg_iou = total_iou / num_samples
print(f"\n📊 Validation Results:")
print(f"   IoU Score: {avg_iou:.4f}")
print(f"   Images evaluated: {num_samples}")
print(f"   Results saved to: segmentation_results.png")